In [7]:
import os
import re
import sys

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from company_name_matcher import CompanyNameMatcher  # pip install company-name-matcher

ROOT_PATH = r'D:\Users\wangy\PycharmProjects\GlassDoor'

if ROOT_PATH not in sys.path:
    sys.path.append(ROOT_PATH)

from Constants import Constants as const

In [3]:
glassdoor_path = r'D:\Users\wangy\Documents\data\glassdoor'

In [8]:
gd_df = pd.read_csv(os.path.join(glassdoor_path, 'Glassdoor岗位评价.csv'))

C:\Users\wangy\AppData\Local\Temp\ipykernel_29336\96579926.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  gd_df = pd.read_csv(os.path.join(glassdoor_path, 'Glassdoor岗位评价.csv'))


In [10]:
gd_df.shape

(9901889, 21)

In [11]:
gd_df.columns = [
    'GD_JobTitle', 'GD_CompanyLink', 'GD_CompanyName', 'GD_CompanyID', 'GD_ReviewDate',
    'GD_Rating', 'GD_ReviewTitle', 'GD_ReviewerStatus', 'GD_Pros', 'GD_Cons', 'GD_Advice',
    'GD_Recommend', 'GD_CEOSupport', 'GD_Outlook', 'GD_CareerOpp', 'GD_CompBenefits',
    'GD_Management', 'GD_WorkLife', 'GD_CultureValues', 'GD_Diversity', 'GD_Index'
]


In [13]:
gd_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_raw_glassdoor_reviews.pkl'))

In [15]:
gd_df[gd_df['GD_CompanyID'] == 8131102]

,GD_JobTitle,GD_CompanyLink,GD_CompanyName,GD_CompanyID,GD_ReviewDate,GD_Rating,GD_ReviewTitle,GD_ReviewerStatus,GD_Pros,GD_Cons,...,GD_Recommend,GD_CEOSupport,GD_Outlook,GD_CareerOpp,GD_CompBenefits,GD_Management,GD_WorkLife,GD_CultureValues,GD_Diversity,GD_Index
1692809,Marketing and Sales Associate,Reviews/Werner-Elektrik-Reviews-E8131102.htm,Werner Elektrik,8131102,2022-12-08,4.0,Good experience,Current Employee,Flexible working hours Work from home,Low salary Inter team communication,...,v,o,r,4.0,3.0,4.0,3.0,3.0,5.0,NaN


In [16]:
mapping_df = pd.read_csv(os.path.join(glassdoor_path, 'wxr_data', 'glassdoor_to_wrds_mapping_filtered.csv'))

In [18]:
gd_df = gd_df.merge(mapping_df.drop(['GD_CompanyName', 'similarity'], axis=1), on='GD_CompanyID', how='left')
gd_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_reviews_with_gvkey.pkl'))

In [21]:
gd_df['GD_ReviewDate'] = pd.to_datetime(gd_df['GD_ReviewDate'])
gd_df[const.YEAR] = gd_df['GD_ReviewDate'].dt.year

In [4]:
ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))
ctat_df['datadate'] = pd.to_datetime(ctat_df['datadate'])
ctat_df[const.YEAR] = ctat_df['fyear'].fillna(ctat_df['datadate'].apply(lambda x: x.year if x.month >= 6 else x.year - 1))
ctat_df.drop_duplicates(subset=[const.GVKEY, const.YEAR], keep='last', inplace=True)

C:\Users\wangy\AppData\Local\Temp\ipykernel_9344\2800178292.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  ctat_df = pd.read_csv(os.path.join(const.COMPUSTAT_PATH, '1950_2024_ctat_firm_identifiers.zip'))


In [5]:
ctat_df = ctat_df[ctat_df[const.YEAR] >= 2008].copy()
ctat_df.head()

,costat,curcd,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,...,city,conml,ipodate,loc,phone,sic,state,weburl,fyear,year
96,A,USD,STD,INDL,C,AIR,2009-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2008,2008
97,A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2009,2009
99,A,USD,STD,INDL,C,AIR,2011-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2010,2010
100,A,USD,STD,INDL,C,AIR,2012-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2011,2011
101,A,USD,STD,INDL,C,AIR,2013-05-31,1004,AAR CORP,000361105,...,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2012,2012


In [28]:
gd_df_firm_year = gd_df[['GD_CompanyID', 'GD_CompanyLink', 'GD_CompanyName', const.YEAR, const.GVKEY]].drop_duplicates(
    subset=['GD_CompanyID', const.YEAR], keep='last')
merged_df = gd_df_firm_year.merge(ctat_df[[const.GVKEY, const.YEAR, 'conm', 'conml']], how='left', on=[const.GVKEY, const.YEAR])

In [29]:
merged_df.shape

(223433, 7)

In [34]:
merged_df[merged_df['conm'].notnull() & merged_df[const.GVKEY].notnull()].shape

(20875, 7)

In [38]:
merged_df.loc[merged_df['conm'].isna() & merged_df[const.GVKEY].notnull(), const.GVKEY] = np.nan

In [41]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids.pkl'))

In [39]:
print(ctat_df.head().to_csv(index=False))

costat,curcd,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,cik,add1,add2,addzip,city,conml,ipodate,loc,phone,sic,state,weburl,fyear,year
A,USD,STD,INDL,C,AIR,2009-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2008,2008
A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2009,2009
A,USD,STD,INDL,C,AIR,2011-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2010,2010
A,USD,STD,INDL,C,AIR,2012-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 1100 North Wood Dale Road",,60191,Wood Dale,AAR Corp,1972-04-24,USA,630 227 2000,5080,IL,www.aarcorp.com,2011,2011
A,USD,STD,INDL,C,AIR,2013-05-31,1004,AAR CORP,000361105,1750.0,"One AAR Place, 11

In [6]:
merged_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids.pkl'))

In [8]:
def _default_preprocess(name: str) -> str:
    """
    Conservative normalization for company names.
    Keep it light—embedding models already handle a lot.
    """
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    s = str(name).strip().lower()
    s = re.sub(r"\s+", " ", s)
    # Optional: remove some punctuation that often varies across sources
    s = re.sub(r"[^\w\s&.-]", "", s)
    return s.strip()


def _parse_find_matches_output(matches):
    """
    company-name-matcher's find_matches() returns a list of matches;
    handle common shapes defensively:
      - [("Apple Inc", 0.91), ...]
      - [{"match": "Apple Inc", "score": 0.91}, ...]
      - [{"company": "Apple Inc", "similarity": 0.91}, ...]
      - ["Apple Inc", ...]  (rare)
    Returns: (best_name, best_score) or (None, None)
    """
    if not matches:
        return None, None

    m0 = matches[0]

    # tuple/list: (name, score)
    if isinstance(m0, (tuple, list)) and len(m0) >= 2:
        return m0[0], float(m0[1])

    # dict-like
    if isinstance(m0, dict):
        name = m0.get("match") or m0.get("company") or m0.get("name")
        score = m0.get("score") or m0.get("similarity")
        try:
            score = float(score) if score is not None else None
        except Exception:
            score = None
        return name, score

    # plain string
    if isinstance(m0, str):
        return m0, None

    return None, None


def fuzzy_fill_gvkey_conm_conml(
    merged_df: pd.DataFrame,
    ctat_df: pd.DataFrame,
    *,
    merged_name_col: str = "GD_CompanyName",
    merged_year_col: str = "year",
    ctat_year_col: str = "year",
    ctat_name_col: str = "conml",          # use conml for matching per your rule
    fallback_ctat_name_col: str = "conm",  # if conml missing
    threshold: float = 0.80,
    k: int = 1,
    use_approx: bool = False,
    n_clusters: int = 50,
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
) -> pd.DataFrame:
    """
    Rules implemented:
      1) If gvkey is not null -> skip row
      2) If gvkey is null -> fuzzy match using Compustat name (ctat_df conml; fallback conm)
      3) Only match within the same year
      4) If matched -> fill (gvkey, conm, conml) from ctat_df

    Notes:
      - Builds a separate vector index per year (fast enough for typical sizes).
      - If your ctat_df is huge, consider setting use_approx=True and tune n_clusters.
    """
    out = merged_df.copy()

    # Ensure year is integer for reliable grouping/joining
    out[merged_year_col] = pd.to_numeric(out[merged_year_col], errors="coerce").astype("Int64")
    ctat = ctat_df.copy()
    ctat[ctat_year_col] = pd.to_numeric(ctat[ctat_year_col], errors="coerce").astype("Int64")

    # Build a matcher (preprocess_fn keeps names consistent across both sides)
    matcher = CompanyNameMatcher(model_name, preprocess_fn=_default_preprocess)

    # Work only on rows where gvkey is missing
    missing_mask = out["gvkey"].isna()
    if missing_mask.sum() == 0:
        return out

    # Precompute a per-year Compustat candidate table
    # Use conml for matching; fallback to conm if conml missing
    ctat["_match_name_raw"] = ctat[ctat_name_col]
    ctat.loc[ctat["_match_name_raw"].isna(), "_match_name_raw"] = ctat[fallback_ctat_name_col]
    ctat["_match_name_raw"] = ctat["_match_name_raw"].astype(str)

    # For lookup after matching: (year, normalized_name) -> row (gvkey, conm, conml)
    # If duplicates exist, keep first occurrence.
    ctat["_match_name_norm"] = ctat["_match_name_raw"].map(_default_preprocess)
    ctat_lookup = (
        ctat.dropna(subset=[ctat_year_col])
            .drop_duplicates(subset=[ctat_year_col, "_match_name_norm"])
            .set_index([ctat_year_col, "_match_name_norm"])[["gvkey", "conm", "conml"]]
    )

    # Iterate by year for year-restricted matching
    years_to_process = out.loc[missing_mask, merged_year_col].dropna().unique().tolist()

    for y in years_to_process:
        # Candidate Compustat names for this year
        ctat_y = ctat.loc[ctat[ctat_year_col] == y, ["_match_name_raw", "_match_name_norm"]].dropna()
        if ctat_y.empty:
            continue

        candidates_raw = ctat_y["_match_name_raw"].tolist()

        # Build an index for this year
        # If use_approx=True, n_clusters should scale with number of candidates.
        matcher.build_index(
            candidates_raw,
            n_clusters=min(n_clusters, max(2, int(len(candidates_raw) ** 0.5))) if use_approx else 20,
            save_dir=None,
        )

        # Rows in merged_df for this year needing fill
        idxs = out.index[missing_mask & (out[merged_year_col] == y)]
        if len(idxs) == 0:
            continue

        for idx in idxs:
            query_name = out.at[idx, merged_name_col]
            if pd.isna(query_name) or str(query_name).strip() == "":
                continue

            matches = matcher.find_matches(
                query_name,
                threshold=threshold,
                k=k,
                use_approx=use_approx,
            )

            best_name, best_score = _parse_find_matches_output(matches)
            if not best_name:
                continue

            # Lookup by (year, normalized matched name)
            key = (y, _default_preprocess(best_name))
            if key not in ctat_lookup.index:
                continue

            gvkey_val, conm_val, conml_val = ctat_lookup.loc[key, ["gvkey", "conm", "conml"]].tolist()

            # Fill only missing fields (per your rule: replace missing gvkey/conm/conml)
            if pd.isna(out.at[idx, "gvkey"]):
                out.at[idx, "gvkey"] = gvkey_val
            if pd.isna(out.at[idx, "conm"]) or str(out.at[idx, "conm"]).strip() == "":
                out.at[idx, "conm"] = conm_val
            if pd.isna(out.at[idx, "conml"]) or str(out.at[idx, "conml"]).strip() == "":
                out.at[idx, "conml"] = conml_val

            # Optional: store match diagnostics
            # out.at[idx, "match_score"] = best_score
            # out.at[idx, "matched_name"] = best_name

    return out

In [9]:
updated_merged_df = fuzzy_fill_gvkey_conm_conml(
    merged_df,
    ctat_df,
    threshold=0.85,   # tune
    k=1,
    use_approx=False  # set True if ctat_df per year is very large
)

d:\anaconda3\envs\GlassDoor\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wangy\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

In [12]:
updated_merged_df[updated_merged_df['gvkey'].notnull()].shape

(45528, 7)

In [16]:
updated_merged_df.to_pickle(os.path.join(const.TEMP_PATH, '2008_2023_glassdoor_names_ids_fuzzy_filled.pkl'))